In [1]:
import numpy as np
from pathlib import Path
from scipy.io import savemat

from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
import gymnasium as gym
from gymnasium import spaces

# ============================================================
# Paths
# ============================================================

PROJECT_ROOT = Path(r"m:\AIforCyberSecurity\Project")

model_path = PROJECT_ROOT / "models" / "sac_lam2_005" / "sac_model.zip"
vecnorm_path = PROJECT_ROOT / "models" / "sac_lam2_005" / "vecnormalize.pkl"

export_path = PROJECT_ROOT / "models" / "sac_lam2_005" / "sac_actor_export.mat"

# ============================================================
# Feature order used during training
# ============================================================

feature_cols = [
    "sigma_rssi_bar",
    "sigma_gps_bar",
    "snr_bar",
    "plr_bar",
    "relative_speed_bar",
    "relative_height_bar",
    "trusted_count",
    "trust_bar",
    "theta"
]

obs_dim = len(feature_cols)

# ============================================================
# Dummy env only for loading VecNormalize + model
# ============================================================

class DummyUAVEnv(gym.Env):
    def __init__(self):
        super().__init__()

        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(obs_dim,),
            dtype=np.float32
        )

        self.action_space = spaces.Box(
            low=np.array([-2.0], dtype=np.float32),
            high=np.array([2.0], dtype=np.float32),
            dtype=np.float32
        )

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        return np.zeros(obs_dim, dtype=np.float32), {}

    def step(self, action):
        return np.zeros(obs_dim, dtype=np.float32), 0.0, False, False, {}

base_env = DummyVecEnv([lambda: DummyUAVEnv()])

vec_env = VecNormalize.load(str(vecnorm_path), base_env)
vec_env.training = False
vec_env.norm_reward = False

model = SAC.load(str(model_path), env=vec_env)

print("Model loaded successfully.")

# ============================================================
# Inspect actor network
# ============================================================

actor = model.policy.actor

print("\n===== ACTOR NETWORK =====")
print(actor)

print("\n===== ACTOR STATE DICT KEYS =====")
for k, v in actor.state_dict().items():
    print(k, tuple(v.shape))

# ============================================================
# Extract actor weights
# For SB3 SAC MlpPolicy, deterministic action uses:
# obs_norm -> latent_pi -> mu -> tanh -> action scaling
# ============================================================

state_dict = actor.state_dict()

export_dict = {}

for key, tensor in state_dict.items():
    clean_key = key.replace(".", "_")
    export_dict[clean_key] = tensor.detach().cpu().numpy()

# ============================================================
# Export VecNormalize observation normalization stats
# obs_norm = clip((obs - mean) / sqrt(var + epsilon), -clip_obs, clip_obs)
# ============================================================

export_dict["obs_mean"] = vec_env.obs_rms.mean
export_dict["obs_var"] = vec_env.obs_rms.var
export_dict["obs_epsilon"] = np.array([vec_env.epsilon])
export_dict["clip_obs"] = np.array([vec_env.clip_obs])

# ============================================================
# Export action scaling
# Your action space is [-2, 2]
# SB3 actor gives squashed action in [-1, 1]
# env_action = action_bias + action_scale * tanh(mu)
# For [-2, 2], action_bias = 0, action_scale = 2
# ============================================================

action_low = np.array([-2.0])
action_high = np.array([2.0])

action_scale = (action_high - action_low) / 2.0
action_bias = (action_high + action_low) / 2.0

export_dict["action_low"] = action_low
export_dict["action_high"] = action_high
export_dict["action_scale"] = action_scale
export_dict["action_bias"] = action_bias

# Metadata
export_dict["feature_order"] = np.array(feature_cols, dtype=object)

# ============================================================
# Save to MATLAB .mat file
# ============================================================

savemat(str(export_path), export_dict)

print("\nSaved MATLAB actor export:")
print(export_path)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Model loaded successfully.

===== ACTOR NETWORK =====
Actor(
  (features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (latent_pi): Sequential(
    (0): Linear(in_features=9, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
  )
  (mu): Linear(in_features=256, out_features=1, bias=True)
  (log_std): Linear(in_features=256, out_features=1, bias=True)
)

===== ACTOR STATE DICT KEYS =====
latent_pi.0.weight (256, 9)
latent_pi.0.bias (256,)
latent_pi.2.weight (256, 256)
latent_pi.2.bias (256,)
mu.weight (1, 256)
mu.bias (1,)
log_std.weight (1, 256)
log_std.bias (1,)

Saved MATLAB actor export:
m:\AIforCyberSecurity\Project\models\sac_lam2_005\sac_actor_export.mat


In [3]:
!python --version

Python 3.10.19


In [4]:
from pathlib import Path

export_path = Path(r"m:\AIforCyberSecurity\Project\models\sac_lam2_005\sac_actor_export.mat")

print(export_path)
print("Exists:", export_path.exists())
print("Size MB:", export_path.stat().st_size / (1024 * 1024) if export_path.exists() else None)

m:\AIforCyberSecurity\Project\models\sac_lam2_005\sac_actor_export.mat
Exists: True
Size MB: 0.2647552490234375
